In [ ]:
!pip install pyspark


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [4]:
from pyspark.sql import SparkSession
# from pyspark.sql.functions import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.window import Window

In [5]:
spark = SparkSession.builder.appName("MySparkApp").getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/30 15:45:39 WARN Utils: Your hostname, admins-MacBook-Pro-9.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.6 instead (on interface en0)
26/09/30 15:45:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/30 15:45:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [6]:
activity_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("activity_type", StringType(), True),
    StructField("distance_km", DoubleType(), True),
    StructField("duration_min", DoubleType(), True),
    StructField("calories_burned", IntegerType(), True),
    StructField("start_time", StringType(), True),
])

profile_schema = StructType([
    StructField("user_id", StringType(), True),
    StructField("age", IntegerType(), True),
    StructField("gender", StringType(), True),
    StructField("country", StringType(), True),
])

weather_schema = StructType([
    StructField("location", StringType(), True),
    StructField("temperature_celsius", IntegerType(), True),
    StructField("weather", StringType(), True),
    StructField("timestamp", StringType(), True),
])


In [14]:
activity_df = spark.read.schema(activity_schema).option("multiLine", True).json("data/activity.json")
profile_df = spark.read.schema(profile_schema).option("multiLine", True).json("data/profile_user.json")
weather_df = spark.read.schema(weather_schema).option("multiLine", True).json("data/weather.json")

In [16]:
activity_df.show()
profile_df.show()
weather_df.show()

+----------+-------------+-----------+------------+---------------+--------------------+
|   user_id|activity_type|distance_km|duration_min|calories_burned|          start_time|
+----------+-------------+-----------+------------+---------------+--------------------+
|1234567890|      running|        5.2|        30.0|            400|2023-06-01T07:00:00Z|
|2345678901|      cycling|       15.0|        45.0|            600|2023-06-01T07:30:00Z|
|3456789012|      walking|        3.0|        25.0|            200|2023-06-01T08:00:00Z|
|1234567890|     swimming|        1.0|        20.0|            300|2023-06-01T08:00:00Z|
+----------+-------------+-----------+------------+---------------+--------------------+

+----------+---+------+-------+
|   user_id|age|gender|country|
+----------+---+------+-------+
|1234567890| 25|female|    USA|
|2345678901| 30|  male| Canada|
|3456789012| 22|female|     UK|
+----------+---+------+-------+

+--------+-------------------+-------+--------------------+
|l

In [19]:
from pyspark.sql import functions as F



df = activity_df.withColumn("start_time", F.to_timestamp("start_time"))\
    .join(
        profile_df.select("user_id", "country"),
        on="user_id",
        how="inner"
    )



weather = (
    weather_df
    .withColumn("timestamp", F.to_timestamp("timestamp"))
    .select(
        F.col("location").alias("country"),
        F.col("timestamp").alias("start_time"),
        "weather"
    )
)

df = df.join(
    weather,
    on=["country", "start_time"],
    how="inner"
)

result = (
    df
    .withColumn("activity_date", F.to_date("start_time"))
    .groupBy("activity_date", "activity_type", "country", "weather")
    .agg(
        F.sum("distance_km").alias("total_distance_km"),
        F.sum("duration_min").alias("total_duration_min")
    )
    .withColumn(
        "average_speed_kmph",
        F.col("total_distance_km") /
        (F.col("total_duration_min") / 60.0)
    )
    .select(
        "activity_type",
        "country",
        "weather",
        "total_distance_km",
        "average_speed_kmph",
        "activity_date"
    )
)

result.show()
result.write.mode("overwrite").partitionBy("activity_date").json("output/activity_summary")

+-------------+-------+-------+-----------------+------------------+-------------+
|activity_type|country|weather|total_distance_km|average_speed_kmph|activity_date|
+-------------+-------+-------+-----------------+------------------+-------------+
|      running|    USA|  sunny|              5.2|              10.4|   2023-06-01|
|      cycling| Canada|  rainy|             15.0|              20.0|   2023-06-01|
|      walking|     UK|  snowy|              3.0| 7.199999999999999|   2023-06-01|
+-------------+-------+-------+-----------------+------------------+-------------+



In [22]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

hourly_counts = activity_df.withColumn(
        "peak_hour",
        F.date_format(
            F.to_timestamp("start_time"),
            "HH"
        )
    )\
    .groupBy(
        "activity_type",
        "peak_hour"
    )\
    .agg(
        F.count("*").alias("activity_count")
    )

w = Window.partitionBy("activity_type").orderBy(
    F.col("activity_count").desc()
)

result = hourly_counts\
    .withColumn(
        "rank",
        F.dense_rank().over(w)
    )\
    .filter(F.col("rank") == 1)\
    .select(
        "activity_type",
        "peak_hour",
        "activity_count"
    )


result.show()
result.write.mode("overwrite").json("output/peak_hour_activity_count")

+-------------+---------+--------------+
|activity_type|peak_hour|activity_count|
+-------------+---------+--------------+
|      cycling|       13|             1|
|      running|       12|             1|
|     swimming|       13|             1|
|      walking|       13|             1|
+-------------+---------+--------------+



In [25]:


user_activity_days_df=activity_df\
    .select(
        "user_id",
        F.to_date(F.to_timestamp("start_time")).alias("activity_date")
    )\
    .dropDuplicates(["user_id", "activity_date"])


user_activity_country_df = user_activity_days_df \
    .join(
        profile_df.select("user_id", "country"),
        on="user_id",
        how="inner"
    )


user_window = Window \
    .partitionBy("user_id") \
    .orderBy("activity_date")


streak_group_df=user_activity_country_df.withColumn("rn",F.row_number().over(user_window)).withColumn("grp",
        F.date_sub(F.col("activity_date"),F.col("rn").cast("int")))



user_streak_df=streak_group_df.groupBy("user_id","country","grp").agg(
        F.count("*").cast("int").alias("streak"))



user_max_streak_df = (
    user_streak_df
    .groupBy(
        "user_id",
        "country"
    )
    .agg(
        F.max("streak").alias("streak")
    )
)


country_window = (
    Window
    .partitionBy("country")
    .orderBy(F.col("streak").desc())
)


longest_streak_df = (
    user_max_streak_df
    .withColumn(
        "rank",
        F.dense_rank().over(country_window)
    )
    .filter(F.col("rank") == 1)
    .select(
        "user_id",
        "country",
        "streak"
    )
)

longest_streak_df.show()

longest_streak_df.write.mode("overwrite").json("output/longest_streak")


+----------+-------+------+
|   user_id|country|streak|
+----------+-------+------+
|2345678901| Canada|     1|
|3456789012|     UK|     1|
|1234567890|    USA|     1|
+----------+-------+------+



In [26]:
user_activity_stats_df = activity_df.groupBy("user_id","activity_type") \
    .agg(
        F.sum("duration_min").alias("total_duration_min"),
        F.count("*").alias("activity_count")
    )

activity_country_df = user_activity_stats_df.join(profile_df.select("user_id", "country"), on="user_id", how="inner")

country_avg_duration_df = activity_country_df.groupBy("activity_type","country")\
    .agg(F.sum("total_duration_min").alias("total_duration_min"),
        F.sum("activity_count").alias("activity_count"))\
    .withColumn("average_duration_min",F.col("total_duration_min") / F.col("activity_count"))\
    .select("activity_type","country","average_duration_min")

rank_window = Window.partitionBy("activity_type").orderBy(F.col("average_duration_min").desc())


ranked_country_df = country_avg_duration_df.withColumn("rank",F.dense_rank().over(rank_window))\
    .filter(F.col("rank") <= 3)


top_countries_df = ranked_country_df.groupBy("activity_type")\
    .agg(
        F.sort_array(
            F.collect_list(
                F.struct(
                    "rank",
                    "country",
                    "average_duration_min"
                )
            )
        ).alias("ranked_countries")
    )\
    .withColumn(
        "top_countries",
        F.transform(
            "ranked_countries",
            lambda x: F.struct(
                x["country"].alias("country"),
                x["average_duration_min"].alias(
                    "average_duration_min"
                )
            )
        )
    )\
    .select(
        "activity_type",
        "top_countries"
    )


top_countries_df.show(truncate=False)

+-------------+----------------+
|activity_type|top_countries   |
+-------------+----------------+
|cycling      |[{Canada, 45.0}]|
|running      |[{USA, 30.0}]   |
|swimming     |[{USA, 20.0}]   |
|walking      |[{UK, 25.0}]    |
+-------------+----------------+



In [27]:
top_countries_df.write.mode("overwrite").json("output/top_countries")